# Exercise 04: Image Registration
**AI in Medical Robotics, WS 2026/27.** Lecture 03: Medical Imaging and AI.

Before a brain operation, the surgeon plans the target on an MRI. During the operation, a robot must know where that target is *now*, in the head on the table. Bringing the two images together is **registration**. In this task you do it by hand, then you watch the computer do it, and then you see where it fails.

**How to work (at home, about 30 minutes)**
1. Run the cells from top to bottom: click the play button on the left of a cell, or press Shift+Enter. Grey cells with only a title, such as "Setup", only need to be run. The Setup cell installs one small package by itself; the first time this takes about half a minute.
2. Work through **1.1, 1.2 and 1.3 in this order**. Each part tells you what to run and what to look at in the output.
3. In one cell you write one line of code, marked `YOUR CODE`; a check cell tells you if it works. In one cell you change one number marked `EDIT`.
4. Nothing is submitted or graded. In the session we go through what you saw and ask you a few questions about it.

The table of contents in Colab's left sidebar jumps to a number. No GPU is needed.

In [ ]:
#@title Setup: run this cell first { display-mode: "form" }
import importlib, importlib.util, subprocess, sys
for _pkg in ("nilearn",):                            # the only package that Colab does not have already
    if importlib.util.find_spec(_pkg) is None:
        print(f"installing {_pkg} (one small package, automatic, about half a minute, only the first time) ...", flush=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg], check=True)
importlib.invalidate_caches()
import time, warnings
import numpy as np
from scipy import ndimage as ndi
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from IPython.display import display
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 10, "font.size": 9})
print("Setup done")

---
# Task 1: Registration, can the robot find its target?
**Registration** lays two images of the same patient on top of each other, so that the same anatomical point lies at the same place in both. The image that is moved is the **moving image**; the image it is moved onto is the **fixed image** (lecture slide 83). In brain surgery the moving image is the **preoperative** MRI (taken before the operation) and the fixed image is the **intraoperative** image (taken during the operation).

**How the result is judged.** Nine **landmarks**, placed by hand, mark the same anatomical points in both images. After a perfect registration each landmark lies exactly on its partner; their distance in mm is the registration error at that point. The computer never uses the landmarks. They only judge the result.

**How to read the pictures.** Moving image magenta, fixed image green, grey where both agree. Magenta circles: the landmarks of the moving image. Green crosses: where they should be. One pixel is one millimetre (the slice comes from the MNI152 brain template, Fonov et al., 2009, 2011).

Run the next cell. It loads the image and the computer's tools.

In [ ]:
#@title Load the MRI slice, the landmarks and the computer's tools { display-mode: "form" }
from nilearn.datasets import load_mni152_template
_vol = load_mni152_template(resolution=1).get_fdata()    # the 1 mm template
fixed = np.pad(np.rot90(_vol[:, :, 80]).astype(np.float32), 20); fixed /= fixed.max()   # axial slice, padded only: 1 pixel = 1 mm
H, W = fixed.shape; CENTER = np.array([H / 2, W / 2])
# nine anatomical landmarks (row, column), placed by hand
LANDMARKS = np.array([(50, 118), (98, 106), (98, 132), (168, 98), (168, 140), (222, 118), (140, 50), (140, 188), (75, 160)], float)
CHALLENGES_REG = {0: (12, 8, -6), 1: (-9, 6, 9), 2: (15, -7, -5), 3: (-14, -9, 4), 4: (8, 10, 10),
                  5: (-11, -5, -10), 6: (17, 4, -9), 7: (-13, -10, 7), 8: (10, -8, 8)}

def rotation(angle_deg):
    a = np.deg2rad(angle_deg)
    return np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])

def transform(image, angle, ty, tx):
    """Rigid transformation: rotate the image by angle (degrees) about its centre, then shift it by ty rows (down) and tx columns (right), in mm."""
    center = np.array(image.shape) / 2
    inverse = rotation(angle).T
    return ndi.affine_transform(image, inverse, offset=center - inverse @ (center + np.array([ty, tx])), order=1)

def transform_points(points, angle, ty, tx):
    """Move landmarks (rows of (row, column)) exactly like transform() moves the image."""
    return (points - CENTER) @ rotation(angle).T + CENTER + np.array([ty, tx])

def make_moving(challenge=0):
    """A second scan of the same head in another position, with a little scanner noise."""
    p = CHALLENGES_REG[challenge]
    noise = np.random.default_rng(challenge).normal(0, 0.02, fixed.shape).astype(np.float32)
    return transform(fixed, *p) + noise, transform_points(LANDMARKS, *p)

def landmark_errors(points_a, points_b):
    """Distance in mm between corresponding landmarks, one number per landmark."""
    return np.linalg.norm(points_a - points_b, axis=1)

def mean_landmark_error(points_a, points_b):
    return float(landmark_errors(points_a, points_b).mean())

def msd(a, b):
    """Image similarity: the mean squared difference of all pixels (smaller = more alike)."""
    return float(((a - b) ** 2).mean())

def show_overlay(ax, a, b, title, points_a=None, points_b=None):
    """a in magenta, b in green: where both agree, the image is grey."""
    ax.imshow(np.clip(np.dstack([a, b, a]), 0, 1)); ax.set_title(title); ax.axis("off")
    if points_a is not None: ax.plot(points_a[:, 1], points_a[:, 0], "o", mfc="none", mec="magenta", ms=6)
    if points_b is not None: ax.plot(points_b[:, 1], points_b[:, 0], "+", c="lime", ms=8, mew=1.5)

_msd_reference = msd   # kept aside: the check cell compares your measure with it

print("MRI slice and landmarks loaded")

## 1.1 Align by hand
A second scan of the same head was taken in another position. The head moved, but it did not change shape. You do the registration by hand first; in 1.2 you watch the computer do the same.

**What you do**
1. In the next cell, set `CHALLENGE` to the last digit of your student number (0 to 8; for 9 take 1) and run it. You see the moving image (magenta) on the fixed image (green). If you change the digit later, run the cells below it again (Runtime, Run after).
2. Run the cell **Align by hand**. Move the moving image with the three sliders until magenta lies on green. The buttons next to each slider change the value by 1 or by 0.25, for fine alignment. The title shows the landmark error in mm. Landmark error: every magenta circle has a partner, the green cross at the same anatomical point. The distance between circle and cross, in mm, is the error at that landmark. The title shows the mean of the nine distances. Zero would mean every circle sits exactly on its cross. Stop when you cannot improve it.

Note what you had to find: one angle and two shifts. That is all a rigid registration is.

In [ ]:
CHALLENGE = 0   # EDIT: the last digit of your student number (0 to 8; for 9 take 1)

moving, moving_landmarks = make_moving(CHALLENGE)
print(f"challenge {CHALLENGE}: landmark error before registration {mean_landmark_error(moving_landmarks, LANDMARKS):.1f} mm")
fig, ax = plt.subplots(figsize=(4.6, 5)); show_overlay(ax, moving, fixed, "moving (magenta) on fixed (green)", moving_landmarks, LANDMARKS); plt.show()

In [ ]:
#@title Align by hand: sliders, and buttons for fine steps { display-mode: "form" }
def _control(description, lo, hi):
    """A slider plus four buttons: -1, -0.25, +0.25, +1."""
    slider = widgets.FloatSlider(min=lo, max=hi, step=0.25, value=0, description=description, continuous_update=False,
                                 readout_format=".2f", style={"description_width": "110px"}, layout=widgets.Layout(width="470px"))
    buttons = []
    for d in (-1, -0.25, 0.25, 1):
        b = widgets.Button(description=f"{d:+g}", layout=widgets.Layout(width="58px"))
        b.on_click(lambda _, d=d: setattr(slider, "value", float(np.clip(slider.value + d, lo, hi))))
        buttons.append(b)
    return slider, widgets.HBox([slider, *buttons], layout=widgets.Layout(flex_flow="row wrap"))

def _align(angle, ty, tx):
    aligned = transform(moving, angle, ty, tx); lm = transform_points(moving_landmarks, angle, ty, tx)
    fig, ax = plt.subplots(figsize=(5.2, 5.6))
    show_overlay(ax, aligned, fixed, f"challenge {CHALLENGE}: angle {angle:+.2f}°, down {ty:+.2f}, right {tx:+.2f} mm\nlandmark error {mean_landmark_error(lm, LANDMARKS):.2f} mm", lm, LANDMARKS)
    plt.show()

_s_angle, _row_angle = _control("angle (°)", -25, 25)
_s_ty, _row_ty = _control("shift down (mm)", -20, 20)
_s_tx, _row_tx = _control("shift right (mm)", -20, 20)
display(widgets.VBox([_row_angle, _row_ty, _row_tx]), widgets.interactive_output(_align, {"angle": _s_angle, "ty": _s_ty, "tx": _s_tx}))

## 1.2 How the computer does it: the algorithm
You aligned the images with your eyes. The computer has only pixel values. Every registration algorithm therefore has three parts (lecture slide 83):

* **A transformation.** Here: one angle and two shifts, the same three numbers you moved by hand.
* **A similarity measure.** One number that says how well two images match. Ours is the **mean squared difference**: subtract the images pixel by pixel, square, take the mean. It is small when the same tissue lies at the same place.
* **A search.** Start with no movement. In every **iteration**, change each of the three numbers a little up and a little down and keep the change that makes the images most alike. When nothing helps, halve the step, down to 0.25.

The transformation and the search are given. The similarity measure you write yourself: it is the part that defines what "aligned" means to the computer. The landmarks are not used for any of this; they only check the result afterwards.


**What you do**
1. In the cell marked `YOUR CODE`, write `mean_squared_difference(a, b)`: it gets two images of the same size and returns one number. Subtract them pixel by pixel, square the differences, take the mean over all pixels. One line. Example: a = [[0, 1], [1, 0]] and b = [[1, 1], [0, 0]] give 0.5. Run it and the check cell; if the check fails, the notebook uses a reference version and you can continue.
2. Read the cell **The computer's search** (twelve lines, with comments) and run it.
3. Run **The computer's registration** and click through the iterations with the buttons: left the images, right the graph with the mean squared difference the computer sees (blue) and the landmark error it never sees (red).

The registration is done. In the session we ask whether this would be enough for robotic brain surgery; 1.3 shows what happens in a real head.

In [ ]:
def mean_squared_difference(a, b):
    """Similarity measure: subtract the two images pixel by pixel, square the differences, take the mean over all pixels. Smaller = more alike."""
    return None   # YOUR CODE (home): replace None by your one line

In [ ]:
#@title Check your similarity measure { display-mode: "form" }
_a = np.array([[0.0, 1.0], [1.0, 0.0]]); _b = np.array([[1.0, 1.0], [0.0, 0.0]])
try:
    _v = mean_squared_difference(_a, _b)
    _ok = (_v is not None and np.isclose(float(_v), 0.5) and np.isclose(float(mean_squared_difference(fixed, fixed)), 0.0)
           and np.isclose(float(mean_squared_difference(moving, fixed)), _msd_reference(moving, fixed)))
except Exception:
    _ok = False
if _ok:
    msd = mean_squared_difference
    print("your similarity measure works; the computer uses it from now on.")
else:
    msd = _msd_reference
    print("not yet: the example should give 0.5, and an image compared with itself gives 0. The notebook uses the reference version, so you can continue.")
print(f"moving on fixed, before registration: {msd(moving, fixed):.5f}   fixed on itself: {msd(fixed, fixed):.5f}")

In [ ]:
# The computer's search. Read it once and make sure you can say in your own words what one iteration does
# (try small changes of the three numbers, keep the one that makes the images more alike, halve the step when nothing helps).
# You do not need to understand every Python detail; if a line is unclear, ask an AI assistant to explain it. Then run the cell.
def register_rigid(moving, fixed, first_step=8, last_step=0.25):
    """Rigid registration: start at no movement, try small changes, keep the best, halve the step.
    Uses only the images, never the landmarks."""
    params = np.zeros(3); score = msd(moving, fixed); step = first_step
    history = [(params.copy(), score, step)]                     # one entry per iteration, for the plots
    while step >= last_step:
        best = None
        for i in range(3):                                       # 0 angle, 1 shift down, 2 shift right
            for sign in (+1, -1):
                candidate = params.copy(); candidate[i] += sign * step
                s = msd(transform(moving, *candidate), fixed)
                if s < score and (best is None or s < best[1]):
                    best = (candidate, s)
        if best is None:
            step = step / 2                                      # nothing helped: try finer steps
            if step < last_step: break                           # finest step reached: done
        else:
            params, score = best
        history.append((params.copy(), score, step))
    return params, history

In [ ]:
#@title The computer's registration: every iteration { display-mode: "form" }
from IPython.display import clear_output
t0 = time.time()
params, history = register_rigid(moving, fixed)
found_landmarks = transform_points(moving_landmarks, *params)
print(f"Challenge {CHALLENGE}: {len(history) - 1} iterations, {time.time() - t0:.1f} s")
print(f"found: angle {params[0]:+.2f}°, shift down {params[1]:+.2f} mm, shift right {params[2]:+.2f} mm")
print(f"landmark error after the registration: {mean_landmark_error(found_landmarks, LANDMARKS):.2f} mm (the computer never saw the landmarks)")
print("\nClick through the iterations with the buttons below. Left: the images. Right: what the computer sees (blue) and what it never sees (red); the dot is the current iteration.")

_sims = [h[1] for h in history]
_errs = [mean_landmark_error(transform_points(moving_landmarks, *h[0]), LANDMARKS) for h in history]
_it = [0]; _last = len(history) - 1
_out = widgets.Output(); _label = widgets.Label()
def _draw():
    k = _it[0]; p, s, st = history[k]; lm = transform_points(moving_landmarks, *p)
    _label.value = f"iteration {k} of {_last}"
    with _out:
        clear_output(wait=True)
        fig, (ax, axg) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1, 1.3]})
        what = "after the registration" if k == _last else f"iteration {k}, step {st:g}"
        show_overlay(ax, transform(moving, *p), fixed, f"{what}\nangle {p[0]:+.2f}°, down {p[1]:+.2f}, right {p[2]:+.2f} mm\n"
                     f"mean squared difference {s:.5f}, landmark error {_errs[k]:.2f} mm", lm, LANDMARKS)
        axg.plot(_sims, "o-", color="tab:blue", lw=2, ms=4, label="mean squared difference (computer sees this)")
        axg.plot(k, _sims[k], "o", color="tab:blue", ms=12)
        axg.set_xlabel("iteration"); axg.set_ylabel("mean squared difference", color="tab:blue"); axg.set_title("iteration by iteration")
        ax2 = axg.twinx(); ax2.plot(_errs, "s--", color="tab:red", lw=2, ms=4, label="landmark error, mm (computer never sees this)")
        ax2.plot(k, _errs[k], "s", color="tab:red", ms=12); ax2.set_ylabel("landmark error (mm)", color="tab:red"); ax2.set_ylim(bottom=0)
        h1, l1 = axg.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels(); axg.legend(h1 + h2, l1 + l2, loc="upper right", fontsize=8)
        plt.tight_layout(); plt.show()
_prev = widgets.Button(description="previous", icon="arrow-left", layout=widgets.Layout(width="110px"))
_next = widgets.Button(description="next", icon="arrow-right", layout=widgets.Layout(width="110px"))
_prev.on_click(lambda _: (_it.__setitem__(0, max(0, _it[0] - 1)), _draw()))
_next.on_click(lambda _: (_it.__setitem__(0, min(_last, _it[0] + 1)), _draw()))
display(widgets.HBox([_prev, _next, _label]), _out); _draw()

## 1.3 Brain shift: rigid, then deformable
What you watched in 1.2 is a rigid registration: one angle and two shifts, the whole image moved as one piece. That is fine as long as the head only moves. In an operation the skull is opened, and the brain itself moves: **brain shift** (lecture slide 84). The ninth landmark is now the surgeon's target, right where the skull is opened.

**What you do**
1. Run the cell **Opening the skull** and click through it. Red star: the target. Fluid drains, the pressure drops, and the brain sags away from the opening, towards the centre; the arrows show this (drawn 2.5 times longer; the simulation uses up to 9 mm, in patients the surface can move by more than 20 mm, Nimsky et al., 2000). Magenta is before, green after, grey where nothing moved.
2. Run the cell **Rigid registration on the opened brain**: the same registration as in 1.2. Red star: where the target really is. Magenta circle: where the registration says it is. Read the mean landmark error (the average over the nine landmarks) and the error at the target.

In [ ]:
#@title Opening the skull: watch the brain sag { display-mode: "form" }
from IPython.display import clear_output
preop = fixed.copy()                                                     # preoperative MRI = moving image
_rr, _cc = np.mgrid[:H, :W]; SITE = LANDMARKS[8].copy()                 # the target, where the skull is opened
_dir = (CENTER - SITE) / np.linalg.norm(CENTER - SITE)                  # tissue sinks away from the opening, towards the centre
_amp = 9.0 * np.exp(-((_rr - SITE[0]) ** 2 + (_cc - SITE[1]) ** 2) / (2 * 28.0 ** 2)) * ndi.binary_dilation(preop > 0.05, iterations=3)
shift_r, shift_c = _amp * _dir[0], _amp * _dir[1]                       # displacement of every pixel in mm

def _sag(t):                                                             # the brain after a fraction t of the full shift
    return ndi.map_coordinates(preop, [_rr - t * shift_r, _cc - t * shift_c], order=1)

def _moved_landmarks(p):
    q = p.copy()
    for _ in range(10):
        q = p + np.c_[ndi.map_coordinates(shift_r, q.T, order=1), ndi.map_coordinates(shift_c, q.T, order=1)]
    return q

HEAD = (-6, 4, 2)                                                       # the head also lies differently
intraop = transform(_sag(1.0), *HEAD)                                    # image during the operation = fixed image
intraop_landmarks = transform_points(_moved_landmarks(LANDMARKS), *HEAD)
k = 8; _star = intraop_landmarks[k]                                      # the target's true position after the shift

print("Yellow star: the surgeon's target, where the skull is opened. Click next: fluid drains and the brain sags away from the opening.")
print("Before in magenta, after in green, grey where nothing moved; arrows drawn 2.5 times longer.")
_T = [0.0, 0.25, 0.5, 0.75, 1.0]; _j = [0]
_outA = widgets.Output(); _labA = widgets.Label()
def _drawA():
    t = _T[_j[0]]; _labA.value = f"shift {9 * t:.2f} mm at the target"
    with _outA:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5.2, 5.6))
        show_overlay(ax, preop, _sag(t), f"brain shift: {9 * t:.2f} mm at the target")
        _s = 10; _m = _amp[::_s, ::_s] > 0.5
        if t > 0:
            ax.quiver(_cc[::_s, ::_s][_m], _rr[::_s, ::_s][_m], t * shift_c[::_s, ::_s][_m], t * shift_r[::_s, ::_s][_m], color="yellow", angles="xy", scale_units="xy", scale=0.4)
        _tgt = _moved_landmarks(SITE[None])[0] * t + SITE * (1 - t)
        ax.plot(_tgt[1], _tgt[0], "*", c="red", mec="white", mew=1, ms=18); plt.show()
_pA = widgets.Button(description="previous", icon="arrow-left", layout=widgets.Layout(width="110px"))
_nA = widgets.Button(description="next", icon="arrow-right", layout=widgets.Layout(width="110px"))
_pA.on_click(lambda _: (_j.__setitem__(0, max(0, _j[0] - 1)), _drawA()))
_nA.on_click(lambda _: (_j.__setitem__(0, min(len(_T) - 1, _j[0] + 1)), _drawA()))
display(widgets.HBox([_pA, _nA, _labA]), _outA); _drawA()

In [ ]:
#@title Rigid registration on the opened brain { display-mode: "form" }
params_bs, history_bs = register_rigid(preop, intraop)                   # the same rigid registration as in 1.2
rigid_result = transform(preop, *params_bs)
rigid_landmarks = transform_points(LANDMARKS, *params_bs)               # where the registration puts the landmarks
per_rigid = landmark_errors(rigid_landmarks, intraop_landmarks)
print(f"rigid registration: {len(history_bs) - 1} iterations; mean landmark error {per_rigid.mean():.1f} mm; at the target {per_rigid[k]:.1f} mm")
print("Left: the preoperative MRI (magenta) laid on the opened brain (green) by the rigid registration. Red star: where the target really is. "
      "Magenta circle: where the registration says it is. Right: the error at each landmark, red: the target, dashed line: the mean.")
fig, (ax, axb) = plt.subplots(1, 2, figsize=(11, 4.8), gridspec_kw={"width_ratios": [1, 1.1]})
show_overlay(ax, rigid_result, intraop, "after the rigid registration", rigid_landmarks, intraop_landmarks)
ax.plot(_star[1], _star[0], "*", c="red", mec="white", mew=1, ms=18)
ax.plot([rigid_landmarks[k, 1], _star[1]], [rigid_landmarks[k, 0], _star[0]], "-", c="red", lw=2)
ax.annotate(f"{per_rigid[k]:.1f} mm", xy=(_star[1], _star[0]), xytext=(_star[1] + 12, _star[0] - 18), color="red", fontsize=11, fontweight="bold")
axb.bar(range(9), per_rigid, color=["tab:red" if i == k else "tab:blue" for i in range(9)])
axb.axhline(per_rigid.mean(), color="k", ls="--", lw=1); axb.text(7.6, per_rigid.mean() + 0.3, f"mean {per_rigid.mean():.1f} mm", ha="right", fontsize=9)
axb.set_xticks(range(9)); axb.set_xticklabels([str(i) for i in range(8)] + ["target"]); axb.set_xlabel("landmark"); axb.set_ylabel("error (mm)")
axb.set_title("error at each landmark")
plt.tight_layout(); plt.show()

**Deformable registration.** A rigid registration has three numbers for the whole image. A **deformable** registration estimates a separate shift for every pixel, the **deformation field** (lecture slide 83), and bends the moving image with it; the bent image is the **warped image** (slide 85). The next cell does this with a classical method (optical flow); networks such as iRegNet (slides 85 and 86) learn the same step from data.

**What you do**
1. Run the cell **Deformable registration**. Left: the deformation field the computer estimated (arrows drawn 2.5 times longer; the small arrows far from the target are small errors of the estimate, about 1 mm, not a real shift). Right: the warped image on the opened brain. Red star: where the target really is. Magenta circle: where the registration now says it is.

In [ ]:
#@title Deformable registration: a shift for every pixel { display-mode: "form" }
print("estimating a shift for every pixel, this takes a few seconds ...", flush=True); t0 = time.time()
from skimage.registration import optical_flow_tvl1
from skimage.transform import warp
flow_r, flow_c = optical_flow_tvl1(intraop, rigid_result, attachment=10, tightness=0.3, num_warp=10, num_iter=30)   # the deformation field
print(f"done in {time.time() - t0:.1f} s")
deformed = warp(rigid_result, np.array([_rr + flow_r, _cc + flow_c]), mode="edge")                                   # the warped image

def _follow(points):                                     # where do the landmarks end up after the warp?
    q = points.copy()
    for _ in range(20):
        q = points - np.c_[ndi.map_coordinates(flow_r, q.T, order=1), ndi.map_coordinates(flow_c, q.T, order=1)]
    return q

deformed_landmarks = _follow(rigid_landmarks)
per_def = landmark_errors(deformed_landmarks, intraop_landmarks)
print(f"at the target:                rigid {per_rigid[k]:.1f} mm, deformable {per_def[k]:.1f} mm")
print(f"mean over all nine landmarks: rigid {per_rigid.mean():.1f} mm, deformable {per_def.mean():.1f} mm")
print("Left: the deformation field, a shift for every pixel (arrows drawn 2.5 times longer). Right: the warped image on the opened brain. "
      "Red star: where the target really is. Magenta circle: where the registration now says it is.")
fig, ax = plt.subplots(1, 2, figsize=(9.4, 4.8))
ax[0].imshow(intraop, cmap="gray"); ax[0].set_title("the deformation field the computer estimated"); ax[0].axis("off")
_s = 8; _inside = ndi.binary_erosion(intraop > 0.05, iterations=4); _m = (_inside & (np.hypot(flow_r, flow_c) > 0.8))[::_s, ::_s]
ax[0].quiver(_cc[::_s, ::_s][_m], _rr[::_s, ::_s][_m], -flow_c[::_s, ::_s][_m], -flow_r[::_s, ::_s][_m], color="yellow", angles="xy", scale_units="xy", scale=0.4)
ax[0].plot(_star[1], _star[0], "*", c="red", mec="white", mew=1, ms=18)
show_overlay(ax[1], deformed, intraop, "after deformable registration (warped image on fixed)", deformed_landmarks, intraop_landmarks)
ax[1].plot(_star[1], _star[0], "*", c="red", mec="white", mew=1, ms=18)
ax[1].plot([deformed_landmarks[k, 1], _star[1]], [deformed_landmarks[k, 0], _star[0]], "-", c="red", lw=2)
ax[1].annotate(f"{per_def[k]:.1f} mm", xy=(_star[1], _star[0]), xytext=(_star[1] + 12, _star[0] - 18), color="red", fontsize=11, fontweight="bold")
plt.tight_layout(); plt.show()
print("This works so well because both images come from one scan. Between a real MRI and an ultrasound image (lecture slides 85 and 86) "
      "the deformable step is much harder.")

## References
* Abraham, A., Pedregosa, F., Eickenberg, M., Gervais, P., Mueller, A., Kossaifi, J., Gramfort, A., Thirion, B., & Varoquaux, G. (2014). Machine learning for neuroimaging with scikit-learn. *Frontiers in Neuroinformatics, 8*, 14. https://doi.org/10.3389/fninf.2014.00014 (nilearn, which loads the template)
* Fitzpatrick, J. M., West, J. B., & Maurer, C. R. (1998). Predicting error in rigid-body point-based registration. *IEEE Transactions on Medical Imaging, 17*(5), 694–702. https://doi.org/10.1109/42.736021 (target registration error)
* Fonov, V. S., Evans, A. C., McKinstry, R. C., Almli, C. R., & Collins, D. L. (2009). Unbiased nonlinear average age-appropriate brain templates from birth to adulthood. *NeuroImage, 47*(Suppl. 1), S102. https://doi.org/10.1016/S1053-8119(09)70884-5
* Fonov, V., Evans, A. C., Botteron, K., Almli, C. R., McKinstry, R. C., Collins, D. L., & Brain Development Cooperative Group. (2011). Unbiased average age-appropriate atlases for pediatric studies. *NeuroImage, 54*(1), 313–327. https://doi.org/10.1016/j.neuroimage.2010.07.033 (the MNI152 template)
* Henrich, P., Mathis-Ullrich, F., & Scheikl, P. M. (2025). LUDO: Low-latency understanding of deformable objects using point cloud occupancy functions. *IEEE Transactions on Robotics, 41*, 4283–4299. https://doi.org/10.1109/TRO.2025.3582837
* Nimsky, C., Ganslandt, O., Cerny, S., Hastreiter, P., Greiner, G., & Fahlbusch, R. (2000). Quantification of, visualization of, and compensation for brain shift using intraoperative magnetic resonance imaging. *Neurosurgery, 47*(5), 1070–1080. https://doi.org/10.1097/00006123-200011000-00008
* Zach, C., Pock, T., & Bischof, H. (2007). A duality based approach for realtime TV-L1 optical flow. In *Pattern Recognition (DAGM 2007)*, LNCS 4713 (pp. 214–223). Springer. https://doi.org/10.1007/978-3-540-74936-3_22 (the deformable registration in 1.3, as implemented in scikit-image)
* Zeineldin, R. A., Karar, M. E., Elshaer, Z., Schmidhammer, M., Coburger, J., Wirtz, C. R., Burgert, O., & Mathis-Ullrich, F. (2021). iRegNet: Non-rigid registration of MRI to interventional US for brain-shift compensation using convolutional neural networks. *IEEE Access, 9*, 147579–147590. https://doi.org/10.1109/ACCESS.2021.3120306